<a href="https://colab.research.google.com/github/krueg-lars-web/DataSciene/blob/main/colab/exercise_02/CURVE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on CURVE

Acknowledgements: [Stefan Oehmcke](https://stefoe.github.io/) for the code used in this notebook.

This notebook will guide you through a data preprocessing pipeline implementing the CURVE framework you heard about in the lecture.

- CURVE: Clean – Unite – Record – Validate – Export
- Dataset: Our World in Data (OWID) CO2 + Energy (open, no API key)
- Question: Is a higher low-carbon electricity share associated with lower CO2 per capita (2000–2024)?

## Initial configuration

Import statements are in the first code cell, so the order in which later cells are executed only depends on what they are doing, not what packages they need (because all the packages are already loaded).

Also placing variables that configure high-level behaviour at the top of your notebooks makes it easier to repurpose a notebook.

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import platform
import matplotlib.pyplot as plt

YEARS = (2000, 2024)
MISSING_POLICY = "drop"         # options: "drop" | "interpolate"
INTERP_LIMIT = 2                # max consecutive years to fill when interpolating
EXPORT_BASENAME = f"owid_co2_energy_ready_{date.today().isoformat()}"

CO2_URL    = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
ENERGY_URL = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"

## Load data

> <font color="#e8710a">**📝 Tasks**</font>
>
> 1. Use [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) to load each of the datasets.

In [7]:
co2_raw = pd.read_csv(CO2_URL)
co2_raw

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.000,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50406,Zimbabwe,2020,ZWE,15526887.0,2.317871e+10,0.496,0.032,8.491,-1.776,-17.298,...,NaN,0.106,0.001,0.001,0.002,0.0,24.146,14.463,1.991,23.450
50407,Zimbabwe,2021,ZWE,15797220.0,2.514009e+10,0.542,0.034,10.223,1.732,20.398,...,NaN,0.105,0.001,0.001,0.002,0.0,27.907,16.408,2.137,20.899
50408,Zimbabwe,2022,ZWE,16069061.0,2.590159e+10,0.387,0.024,12.232,2.009,19.653,...,NaN,0.104,0.001,0.001,0.002,0.0,29.917,18.830,1.380,11.283
50409,Zimbabwe,2023,ZWE,16340829.0,NaN,0.387,0.024,13.443,1.211,9.904,...,NaN,0.103,0.001,0.001,0.002,0.0,31.029,20.318,1.876,13.957


In [8]:
energy_raw = pd.read_csv(ENERGY_URL)
energy_raw

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23372,Zimbabwe,2020,ZWE,15526887.0,2.317871e+10,NaN,0.0,0.0,0.0,6.440,...,0.298,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN
23373,Zimbabwe,2021,ZWE,15797220.0,2.514009e+10,NaN,0.0,0.0,0.0,6.963,...,0.233,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN
23374,Zimbabwe,2022,ZWE,16069061.0,2.590159e+10,NaN,0.0,0.0,0.0,6.845,...,0.336,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN
23375,Zimbabwe,2023,ZWE,16340829.0,NaN,NaN,0.0,0.0,0.0,7.344,...,0.300,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0.0,NaN


# C = Clean

## Select only necessary columns

We don't need all of the many columns that are available.
This selects only a necessary subset of columns.

In [9]:
co2_keep = ["iso_code", "country", "year", "co2_per_capita"]
co2 = co2_raw[co2_keep].copy()
co2

,iso_code,country,year,co2_per_capita
0,AFG,Afghanistan,1750,NaN
1,AFG,Afghanistan,1751,NaN
2,AFG,Afghanistan,1752,NaN
3,AFG,Afghanistan,1753,NaN
4,AFG,Afghanistan,1754,NaN
...,...,...,...,...
50406,ZWE,Zimbabwe,2020,0.547
50407,ZWE,Zimbabwe,2021,0.647
50408,ZWE,Zimbabwe,2022,0.761
50409,ZWE,Zimbabwe,2023,0.823


In [10]:
# Prefer low_carbon_electricity_share; fall back to renewables_share_elec if needed
cands = [c for c in energy_raw.columns if "low_carbon_electricity_share" in c] \
        or [c for c in energy_raw.columns if "renewables_share_elec" in c]
share_col = cands[0]

energy_keep = ["iso_code", "country", "year", share_col, "electricity_demand", "population", "gdp"]
energy = energy_raw[energy_keep].copy()
energy = energy.rename(columns={share_col: "low_carbon_elec_share"})  # whatever share column we end up using, give it a fixed name
energy_keep = ["iso_code", "country", "year", "low_carbon_elec_share", "electricity_demand", "population", "gdp"]
energy


,iso_code,country,year,low_carbon_elec_share,electricity_demand,population,gdp
0,NaN,ASEAN (Ember),2000,6.020,378.76,NaN,NaN
1,NaN,ASEAN (Ember),2001,5.643,405.09,NaN,NaN
2,NaN,ASEAN (Ember),2002,5.363,433.19,NaN,NaN
3,NaN,ASEAN (Ember),2003,5.058,458.30,NaN,NaN
4,NaN,ASEAN (Ember),2004,5.026,496.61,NaN,NaN
...,...,...,...,...,...,...,...
23372,ZWE,Zimbabwe,2020,1.490,8.69,15526887.0,2.317871e+10
23373,ZWE,Zimbabwe,2021,1.284,10.31,15797220.0,2.514009e+10
23374,ZWE,Zimbabwe,2022,1.230,10.85,16069061.0,2.590159e+10
23375,ZWE,Zimbabwe,2023,1.201,11.99,16340829.0,NaN


> <font color="#e8710a">**📝 Tasks**</font>
>
> 2. Looking at the printouts of the dataframes, does the data meet the tidy criteria from the lecture?

**✏️ Your answer**

2.1. Each variable forms a column.  --> erfüllt
2.2. Each observation forms a row. --> erfüllt
2.3. Each value is a cell. --> erfüllt

## Handle missing data

First have a look at where data is missing.
We do this by printing which columns include missing values.

In [11]:
for col in co2_keep:
    has_na = co2[col].notna().all()
    print(f"{col}: {'some missing' if has_na else 'complete'}")

#alternative:
#for col in co2_keep:
    #missing = co2[col].isna().sum()
    #print(f"{col}: {missing} missing values")

iso_code: complete
country: some missing
year: some missing
co2_per_capita: complete


In [12]:
for col in energy_keep:
    has_na = energy[col].notna().all()
    print(f"{col}: {'some missing' if has_na else 'complete'}")

iso_code: complete
country: some missing
year: some missing
low_carbon_elec_share: complete
electricity_demand: complete
population: complete
gdp: complete


> <font color="#e8710a">**📝 Tasks**</font>
>
> 3. Implement the case in which missing data should be dropped.

In [13]:
def handle_missing(data, req_cols):
    # Missing data policy: DROP vs INTERPOLATE
    if MISSING_POLICY == "drop":
        data = data.dropna(subset = req_cols)
        pass
    elif MISSING_POLICY == "interpolate":
        # Interpolate within each country over time (linear), up to INTERP_LIMIT consecutive years
        data = data.sort_values(["iso_code", "year"])

        def _interp_country(g):
            g = g.copy()
            for c in req_cols:
                g[c] = g[c].interpolate(method="linear", limit=INTERP_LIMIT, limit_direction="both")
                # optional: clamp to valid share range after interpolation
                if c == "low_carbon_elec_share":
                    g["low_carbon_elec_share"] = g["low_carbon_elec_share"].clip(lower=0, upper=100)
            return g

        na_before = data[req_cols].isna().sum().sum()
        data = data.groupby("iso_code", group_keys=False).apply(_interp_country)
        na_after = data[req_cols].isna().sum().sum()
        print(f"[Missing policy: INTERPOLATE] NA cells before: {na_before}  after: {na_after}")

        # If some NAs remain (long gaps), drop them so validation is simple for students.
        if na_after > 0:
            rows_before = len(data)
            data = data.dropna(subset=req_cols)
            print(f"Remaining NA rows removed after interpolation: {rows_before - len(data)}")

        print("Comment: OK for short, smooth gaps; document the assumption. Avoid long-gap fills or extrapolation.")
    else:
        raise ValueError("MISSING_POLICY must be 'drop' or 'interpolate'.")

    return data

In [14]:
# cannot interpolate -> drop
co2 = co2.dropna(subset=["iso_code"])
energy = energy.dropna(subset=["iso_code"])

# interpolate if requested
co2 = handle_missing(co2, ["co2_per_capita"])
energy = handle_missing(energy, ["low_carbon_elec_share"])

> <font color="#e8710a">**📝 Tasks**</font>
>
> We only need data for the specified time range, that was configured in the very first code cell of this notebook.
>
> 4. Select rows in both `co2` and `energy` based on `YEARS`.

In [29]:
co2 = co2.query("year >= @YEARS[0] and year <= @YEARS[1]")
energy = energy.query("year >= @YEARS[0] and year <= @YEARS[1]")


In [30]:
print(f"#co2 records: {len(co2)}")  # should be 5325
print(f"#energy records: {len(energy)}")  # should be 5243

#co2 records: 5325
#energy records: 5243


## Inspecting the data

In [31]:
co2.describe()

,year,co2_per_capita
count,5325.00000,5325.000000
mean,2012.00000,5.029754
std,7.21178,6.489615
min,2000.00000,0.021000
25%,2006.00000,0.802000
50%,2012.00000,2.870000
75%,2018.00000,6.770000
max,2024.00000,67.727000


In [32]:
energy.describe()

,year,low_carbon_elec_share,electricity_demand,population,gdp
count,5243.000000,5243.000000,5243.000000,5.243000e+03,3.774000e+03
mean,2012.010299,2.772051,107.143910,3.422386e+07,5.719333e+11
std,7.180661,7.130797,505.747035,1.332922e+08,1.925700e+12
min,2000.000000,0.000000,0.010000,3.119000e+03,3.128536e+08
25%,2006.000000,0.000000,0.930000,9.479885e+05,2.353973e+10
50%,2012.000000,0.045000,7.740000,6.346993e+06,7.815410e+10
75%,2018.000000,1.737500,44.895000,2.289637e+07,3.535943e+11
max,2024.000000,77.586000,10069.880000,1.450936e+09,2.696602e+13


> <font color="#e8710a">**📝 Tasks**</font>
>
> 5. Does this indicate that the correct range of years was selected?
> 6. Is the data roughly equally distributed across the years? (Based on percentiles)

**✏️ Your answer**

5.
6.

# U = UNITE

> <font color="#e8710a">**📝 Tasks**</font>
>
> We want to combine the `co2` and `energy` dataframes by finding matching rows (based on the `iso_code` **and** `year` columns) and merging each match into a new row, creating a new dataframe.
>
> 7. Implement this operation using the [pandas.DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) method and assign it to a new variable called `df`.
>
> Read into the documentation of the `how` parameter to find out which kind of merge operation you need here.

In [ ]:
# TODO
df = None

In [ ]:
print(f"#merged records: {len(df)}")  # should be 4976

# R = RECORD

> <font color="#e8710a">**📝 Tasks**</font>
>
> 8. What are the data sources?
> 9. What major decisions on how to handle the data have been taken?
>     1. Was any data removed?
>     2. Were data types changed?
>     3. Were columns/rows/cells merged or split?
>     4. Were identifiers of columns/rows changed?
>
> A systematic and extended version of this was published in the [ODE protocol](https://www.sciencedirect.com/science/article/pii/S1364815226000599) (Table 1).

**✏️ Your answer**

8.
9.
    1.
    2.
    3.
    4.

# V = VALIDATE

## Check essential properties

> <font color="#e8710a">**📝 Tasks**</font>
>
> 10. Implement an assertion, that checks that there are no repeated ISO codes or years.
> 11. Implement an assertion, that checks that all values in `low_carbon_elec_share` are within the interval [0,100].

In [ ]:
# 1) Keys unique per (iso_code, year)
# TODO

# 2) Required fields not NA
assert ~df[req_cols].isna().any().any(), "Missing values remain in required columns."

# 3) Value ranges
assert (df["co2_per_capita"] >= 0).all(), "Negative CO2 per capita values detected."
# TODO

# 4) Optional plausibility cap (very high extremes rarely occur)
assert df.loc[(df["co2_per_capita"] > 80) | (df["low_carbon_elec_share"] > 100)].empty, "Implausible extremes present."

print("Validation passed ✅")

## Quick sanity outputs

In [ ]:
n_countries = df["iso_code"].nunique()
n_rows = len(df)
print(f"Countries: {n_countries} | Rows: {n_rows} | Years: [{df['year'].min()}–{df['year'].max()}]")

# Simple correlation
corr = df[["co2_per_capita", "low_carbon_elec_share"]].corr().iloc[0,1]
print(f"Pearson corr(co2_per_capita, low_carbon_elec_share) = {corr:.3f}")

# Tiny scatter for a feel (random 3 countries)
sample_c = df["iso_code"].drop_duplicates().sample(n=min(3, n_countries), random_state=0).tolist()
sub = df[df["iso_code"].isin(sample_c)]
plt.figure(figsize=(6,4))
for c in sample_c:
    cc = sub[sub["iso_code"]==c]
    plt.scatter(cc["low_carbon_elec_share"], cc["co2_per_capita"], label=c, s=18, alpha=0.8)
plt.xlabel("Low-carbon electricity share (%)")
plt.ylabel("CO₂ per capita (t/person)")
plt.title("Sample countries across 2000–2024")
plt.legend(title="ISO")
plt.tight_layout()
plt.show()

# E = EXPORT

> <font color="#e8710a">**📝 Tasks**</font>
>
> 12. Save the contents of `df` in CSV format without the index using [pandas.DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).

## Reproduction notes

> <font color="#e8710a">**📝 Tasks**</font>
>
> 13. Print version information for your installation of Python as well as used packages.
> 14. Print configuration values.
> 15. Print column names, that you saved in the previous step.

> <font color="#e8710a">**📝 Tasks**</font>
>
> Check out the [pyproject.toml](https://github.com/bckrlab/data-science-course/blob/main/pyproject.toml) file in the underlying git repository.
>
> 16. How much of this printout is already defined there?
> 17. Research/Discuss: Why is it better to have a file like pyproject.toml compared to the printouts?

**✏️ Your answer**

16.
17.